# Feature Engineering – Customer Churn & Value Modeling
Build customer-level features

Prevent data leakage

Prepare dataset for modeling

> Feature Engineering Overview
This notebook converts EDA insights into stable, customer-level features for churn modeling. All features are computed using historical data prior to a fixed snapshot date to prevent data leakage.

In [17]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/processed/cleaned_transactions.csv")
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])


In [18]:
snapshot_date = df['InvoiceDate'].max() - pd.Timedelta(days=90)


> Features engineered based on EDA insights highlighting engagement, value, and dissatisfaction


In [19]:
customer_df = df.groupby('CustomerID').agg(
    last_purchase_date=('InvoiceDate', 'max'),
    first_purchase_date=('InvoiceDate', 'min'),
    frequency=('InvoiceNo', 'nunique'),
    monetary=('Revenue', 'sum'),
    avg_basket_value=('Revenue', 'mean'),
    total_quantity=('Quantity', 'sum'),
    return_quantity=('Quantity', lambda x: x[x < 0].sum())
).reset_index()


In [24]:
# Recency captures disengagement intensity
# Purchase velocity captures engagement frequency over customer lifetime
# Return ratio captures dissatisfaction behavior identified in EDA
# Recency (days since last purchase at snapshot)
customer_df['recency_days'] = (
    snapshot_date - customer_df['last_purchase_date']
).dt.days.clip(lower=0)

# Customer lifetime
customer_df['customer_lifetime_days'] = (
    customer_df['last_purchase_date'] - customer_df['first_purchase_date']
).dt.days + 1

# Purchase velocity
customer_df['purchase_velocity'] = (
    customer_df['frequency'] / customer_df['customer_lifetime_days']
)

# Return ratio
customer_df['return_ratio'] = (
    customer_df['return_quantity'].abs() / customer_df['total_quantity']
).replace([np.inf, -np.inf], 0).fillna(0)


In [25]:
customer_df['churn'] = (
    customer_df['last_purchase_date'] < snapshot_date
).astype(int)


In [26]:
model_df = customer_df[[
    'CustomerID',                # ← MUST BE HERE
    'frequency',
    'monetary',
    'avg_basket_value',
    'customer_lifetime_days',
    'purchase_velocity',
    'return_ratio',
    'churn'
]]


In [27]:
model_df.isna().sum()
model_df.describe()
model_df['churn'].value_counts(normalize=True)


churn
0    0.667353
1    0.332647
Name: proportion, dtype: float64

In [28]:
model_df.to_csv("data/processed/modeling_dataset.csv", index=False)


> Feature Engineering Summary
The resulting dataset contains customer-level behavioral features aligned with EDA findings and is suitable for downstream modeling and evaluation.